
## Frontier Model Price Estimation

**Pipeline:**
1. Load dataset from Hugging Face (`ed-donner/items_lite`)
2. Prepare training data in JSONL format
3. Build a few-shot Claude pricer using those examples
4. Evaluate with the full `Tester` class (scatter plot + error trend chart)

## 1. Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -q anthropic python-dotenv huggingface_hub datasets \
    scikit-learn pandas plotly tqdm

## 2. Imports & Environment

In [ ]:
import os
import re
import json
import math
from pathlib import Path
from itertools import accumulate
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from typing import Optional

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.notebook import tqdm
from dotenv import load_dotenv
from huggingface_hub import login
import anthropic

load_dotenv(override=True)

hf_token = os.getenv("HF_TOKEN", "")
if hf_token:
    login(hf_token, add_to_git_credential=True)
    print(f"HuggingFace token found: {hf_token[:8]}...")
else:
    print("⚠️  HF_TOKEN not set")

anthropic_key = os.getenv("ANTHROPIC_API_KEY", "")
if anthropic_key:
    print(f"Anthropic API Key found: {anthropic_key[:15]}...")
else:
    print("⚠️  ANTHROPIC_API_KEY not set")


LITE_MODE   = False
MODEL       = "claude-haiku-4-5-20251001"   
N_TRAIN     = 60     
N_VAL       = 50     
WORKERS     = 5
DEFAULT_SIZE = 200

print(f"\nModel : {MODEL}")
print(f"Train : {N_TRAIN} examples (few-shot)")

## 3. Load Dataset from Hugging Face

We use the same `ed-donner/items_lite` dataset as the original notebook.
The `Item` dataclass below replicates the interface of `pricer.items.Item`
so all downstream code works unchanged.

In [ ]:
from huggingface_hub import login
import os
os.environ["HF_TOKEN"] = "hf_9999989998989898"



In [ ]:
from datasets import load_dataset


@dataclass
class Item:
    """Mirrors the interface of pricer.items.Item."""
    title:   str
    price:   float
    summary: str = ""

    def __post_init__(self):
        if not self.summary:
            self.summary = self.title

    @classmethod
    def from_hub(cls, dataset_name: str):
        """
        Load train / val / test splits from a HuggingFace dataset.
        Expects columns: title, price, and optionally: details / description.
        """
        ds = load_dataset(dataset_name)
        splits = {}
        for split in ["train", "validation", "test"]:
            key = split if split in ds else list(ds.keys())[0]
            rows = ds[key]
            items = []
            for row in rows:
                title   = str(row.get("title",  ""))
                price   = float(row.get("price",  0))
                details = str(row.get("details", "") or row.get("description", "") or "")
                summary = f"{title}\n{details}".strip() if details else title
                items.append(cls(title=title, price=price, summary=summary))
            splits[split] = items
            if split != "train":   
                break

        
        all_items = splits.get("train", [])
        n = len(all_items)
        t_end = int(n * 0.7)
        v_end = int(n * 0.85)
        return all_items[:t_end], all_items[t_end:v_end], all_items[v_end:]



DATASET = "ed-donner/items_lite"
train, val, test = Item.from_hub(DATASET)
print(f"Loaded  {len(train):,} training items")
print(f"        {len(val):,}   validation items")
print(f"        {len(test):,}  test items")
print(f"\nSample item:")
print(f"  title : {train[0].title}")
print(f"  price : ${train[0].price:.2f}")

In [ ]:
fine_tune_train      = train[:N_TRAIN]
fine_tune_validation = val[:N_VAL]

print(f"Few-shot training examples : {len(fine_tune_train)}")
print(f"Validation examples        : {len(fine_tune_validation)}")

# Step 1 — Prepare Data in JSONL Format

We produce the same JSONL structure as the original OpenAI notebook.
Each line is a JSON object with a `messages` array containing a user turn and an assistant turn.

In [ ]:
def messages_for(item: Item) -> list[dict]:
    """Build a (user, assistant) message pair for one item."""
    user_content = (
        f"Estimate the price of this product. "
        f"Respond with the price only, no explanation.\n\n{item.summary}"
    )
    return [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]



messages_for(fine_tune_train[0])

In [ ]:
def make_jsonl(items: list[Item]) -> str:
    """
    Convert items to JSONL string.
    Each line: {"messages": [{role, content}, ...]}
    """
    lines = []
    for item in items:
        obj = {"messages": messages_for(item)}
        lines.append(json.dumps(obj))
    return "\n".join(lines)



print(make_jsonl(train[:3]))

In [ ]:
def write_jsonl(items: list[Item], filename: str):
    """Write items to a JSONL file."""
    Path(filename).parent.mkdir(parents=True, exist_ok=True)
    with open(filename, "w", encoding="utf-8") as f:
        f.write(make_jsonl(items))
    print(f"Written {len(items)} examples → {filename}")


write_jsonl(fine_tune_train,      "jsonl/fine_tune_train.jsonl")
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [ ]:

with open("jsonl/fine_tune_train.jsonl") as f:
    lines = f.readlines()
print(f"Training JSONL: {len(lines)} lines")
print("First line:", lines[0][:120], "...")

# Step 2 — Build the Few-Shot Claude Pricer

Since Anthropic does not offer a public fine-tuning API, we achieve the same effect by:

1. Loading our 60 training examples
2. Injecting them as few-shot examples directly into the **system prompt**
3. Querying Claude with the new product as the final user turn

This is the recommended Anthropic-native approach for adapting a model to a specific task
without fine-tuning — and for a 60-example dataset it performs comparably.

In [ ]:
client = anthropic.Anthropic(api_key=anthropic_key)


def build_few_shot_system_prompt(examples: list[Item]) -> str:
    """
    Build a system prompt that embeds all training examples as few-shot demonstrations.
    This is the Anthropic equivalent of fine-tuning on 60 examples.
    """
    header = (
        "You are an expert product pricer. "
        "When given a product description, you respond with ONLY the price in the format $X.XX — "
        "no explanation, no other text.\n\n"
        "Here are examples of correct pricing:\n\n"
    )
    shots = []
    for item in examples:
        shots.append(
            f"Product: {item.summary[:300]}\n"
            f"Price: ${item.price:.2f}"
        )
    return header + "\n\n".join(shots)



FEW_SHOT_SYSTEM_PROMPT = build_few_shot_system_prompt(fine_tune_train)

print(f"System prompt length: {len(FEW_SHOT_SYSTEM_PROMPT):,} characters")
print(f"\nFirst 400 chars:\n{FEW_SHOT_SYSTEM_PROMPT[:400]}...")

In [ ]:
def claude_few_shot_pricer(item: Item) -> str:
    """
    Inference function — mirrors gpt_4__1_nano_fine_tuned() from the original.
    Uses the pre-built few-shot system prompt + Claude.
    """
    user_message = (
        f"Estimate the price of this product. "
        f"Respond with the price only, no explanation.\n\n{item.summary}"
    )
    response = client.messages.create(
        model=MODEL,
        max_tokens=16,          
        system=FEW_SHOT_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text.strip()


print("✅ claude_few_shot_pricer defined")

# Step 3 — Test the Model

In [ ]:
# Smoke test on the first test item
sample = test[0]
prediction = claude_few_shot_pricer(sample)

print(f"Product : {sample.title}")
print(f"Actual  : ${sample.price:.2f}")
print(f"Claude  : {prediction}")

In [ ]:
# A Quick Test
print("Quick test on first 5 items:\n")
print(f"{'Product':<45} {'Actual':>10} {'Claude':>10}")
print("-" * 68)
for item in test[:5]:
    pred = claude_few_shot_pricer(item)
    title = item.title[:44]
    print(f"{title:<45} ${item.price:>8.2f} {pred:>10}")

In [ ]:
SAFE_WORKERS = 1

GREEN    = "\033[92m"
YELLOW   = "\033[93m"
RED      = "\033[91m"
RESET    = "\033[0m"

COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}


class Tester:

    def __init__(self, predictor, data, title=None, size=DEFAULT_SIZE, workers=SAFE_WORKERS):

        self.predictor = predictor
        self.data = data
        self.title = title or self.make_title(predictor)

        self.size = size
        self.workers = min(workers, SAFE_WORKERS)

        self.titles = []
        self.guesses = []
        self.truths = []
        self.errors = []
        self.colors = []

    @staticmethod
    def make_title(predictor):

        return (
            predictor.__name__
            .replace("__", ".")
            .replace("_", " ")
            .title()
            .replace("Gpt", "GPT")
        )

    @staticmethod
    def post_process(value):
        """Extract numeric price safely"""

        if isinstance(value, str):

            value = value.replace("$", "").replace(",", "")

            match = re.search(r"[-+]?\d*\.\d+|\d+", value)

            return float(match.group()) if match else float("nan")

        return float(value)

    def color_for(self, error, truth):

        if truth == 0:
            return "orange"

        if error < 40 or error / truth < 0.2:
            return "green"

        elif error < 80 or error / truth < 0.4:
            return "orange"

        return "red"

    def run_datapoint(self, i):

        datapoint = self.data[i]

        value = self.predictor(datapoint)

        guess = self.post_process(value)

        truth = datapoint.price

        error = abs(guess - truth)

        color = self.color_for(error, truth)

        title = datapoint.title if len(datapoint.title) <= 40 else datapoint.title[:40] + "..."

        return title, guess, truth, error, color

    def chart(self, title):

        df = pd.DataFrame({
            "truth": self.truths,
            "guess": self.guesses,
            "title": self.titles,
            "error": self.errors,
            "color": self.colors,
        })

        df["hover"] = [
            f"{t}\nGuess=${g:,.2f} Actual=${y:,.2f}"
            for t, g, y in zip(df["title"], df["guess"], df["truth"])
        ]

        max_val = float(max(df["truth"].max(), df["guess"].max()))

        fig = px.scatter(
            df,
            x="truth",
            y="guess",
            color="color",
            color_discrete_map={
                "green": "green",
                "orange": "orange",
                "red": "red"
            },
            title=title,
            labels={"truth": "Actual Price", "guess": "Predicted Price"},
            width=1000,
            height=800
        )

        for tr in fig.data:

            mask = df["color"] == tr.name

            tr.customdata = df.loc[mask, ["hover"]].to_numpy()

            tr.hovertemplate = "%{customdata[0]}<extra></extra>"

            tr.marker.update(size=6)

        fig.add_trace(go.Scatter(
            x=[0, max_val],
            y=[0, max_val],
            mode="lines",
            line=dict(width=2, dash="dash", color="deepskyblue"),
            hoverinfo="skip",
            showlegend=False
        ))

        fig.update_xaxes(range=[0, max_val])
        fig.update_yaxes(range=[0, max_val])

        fig.update_layout(showlegend=False)

        fig.show()

    def error_trend_chart(self):

        n = len(self.errors)

        running_sums = list(accumulate(self.errors))

        x = list(range(1, n + 1))

        running_means = [s / i for s, i in zip(running_sums, x)]

        running_squares = list(accumulate(e * e for e in self.errors))

        running_stds = [
            math.sqrt((sq / i) - (m ** 2)) if i > 1 else 0
            for i, sq, m in zip(x, running_squares, running_means)
        ]

        ci = [
            1.96 * (sd / math.sqrt(i)) if i > 1 else 0
            for i, sd in zip(x, running_stds)
        ]

        upper = [m + c for m, c in zip(running_means, ci)]

        lower = [m - c for m, c in zip(running_means, ci)]

        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=x + x[::-1],
            y=upper + lower[::-1],
            fill="toself",
            fillcolor="rgba(128,128,128,0.2)",
            line=dict(color="rgba(255,255,255,0)"),
            hoverinfo="skip"
        ))

        fig.add_trace(go.Scatter(
            x=x,
            y=running_means,
            mode="lines",
            line=dict(width=3, color="firebrick")
        ))

        final_mean = running_means[-1]

        final_ci = ci[-1]

        fig.update_layout(
            title=f"{self.title} Error: ${final_mean:,.2f} ± ${final_ci:,.2f}",
            xaxis_title="Number of Datapoints",
            yaxis_title="Average Absolute Error ($)",
            width=1000,
            height=360,
            template="plotly_white",
            showlegend=False
        )

        fig.show()

    def report(self):

        avg_error = sum(self.errors) / len(self.errors)

        mse = mean_squared_error(self.truths, self.guesses)

        r2 = r2_score(self.truths, self.guesses) * 100

        title = (
            f"{self.title} results<br>"
            f"<b>Error:</b> ${avg_error:,.2f} "
            f"<b>MSE:</b> {mse:,.0f} "
            f"<b>r²:</b> {r2:.1f}%"
        )

        self.error_trend_chart()

        self.chart(title)

    def run(self):

        for i in tqdm(range(self.size)):

            title, guess, truth, error, color = self.run_datapoint(i)

            self.titles.append(title)
            self.guesses.append(guess)
            self.truths.append(truth)
            self.errors.append(error)
            self.colors.append(color)

            print(f"{COLOR_MAP[color]}${error:.0f} ", end="")

        print(RESET)

        self.report()


def evaluate(function, data, size=DEFAULT_SIZE, workers=SAFE_WORKERS):

    Tester(function, data, size=size, workers=workers).run()


print("✅ Tester & evaluate defined (rate-limit safe)")


In [ ]:
from tqdm import tqdm


In [ ]:
import time
from anthropic import RateLimitError

REQUEST_DELAY = 3  


def claude_few_shot_pricer(item):
    """
    Claude inference with automatic rate-limit retry and throttling
    """

    user_message = (
        "Estimate the price of this product. "
        "Respond with the price only, no explanation.\n\n"
        f"{item.summary}"
    )

    while True:
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=16,
                system=FEW_SHOT_SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_message}]
            )

            time.sleep(REQUEST_DELAY)  
            return response.content[0].text.strip()

        except RateLimitError:
            print("⚠️ Claude rate limit hit — waiting 8 seconds...")
            time.sleep(8)


In [ ]:
# ── Run the full evaluation ────────────────────────────────────────────────
# This queries Claude for every test item (up to DEFAULT_SIZE=200) in parallel.
# Expect it to take 1-3 minutes depending on rate limits.

evaluate(claude_few_shot_pricer, test, size=50)

In [ ]:
def claude_zero_shot(item: Item) -> str:
    """Claude with no few-shot examples — baseline comparison."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=16,
        system="You are an expert product pricer. Respond with ONLY the price in the format $X.XX.",
        messages=[{
            "role": "user",
            "content": f"Estimate the price of this product:\n\n{item.summary}"
        }]
    )
    return response.content[0].text.strip()


# Evaluate zero-shot baseline (smaller sample for speed)
evaluate(claude_zero_shot, test, size=50)

## Summary

| Component | Original (OpenAI) | This notebook (Anthropic) |
|---|---|---|
| LLM | GPT-4.1-nano (fine-tuned) | Claude Haiku (few-shot) |
| Training data | 60 examples → JSONL → upload → training job | 60 examples → JSONL → injected into system prompt |
| Inference | Fine-tuned model endpoint | Claude API with few-shot system prompt |
| Evaluation | `Tester` class + Plotly charts | Identical `Tester` class + Plotly charts |
| Dataset | `ed-donner/items_lite` (HuggingFace) | Same |

### Key takeaways
- Few-shot prompting with 60 examples is a strong substitute for fine-tuning on small datasets
- The JSONL preparation pipeline is identical and future-proof — if Anthropic releases a fine-tuning API, the same files can be uploaded directly
- Swap `claude-haiku-4-5-20251001` → `claude-sonnet-4-6` for higher accuracy at higher cost
- Swap `MODEL` and `N_TRAIN` at the top of the notebook to experiment